# Euro 2024 player profiling: who does what, and who plays like whom?

**Question for a coach:** for each position, which players stand out in the jobs that position is asked to do, and if we lose a player, who has a similar profile?

**Data:** StatsBomb open event data for all 51 matches of UEFA Euro 2024 (free, with attribution). Every pass, carry, shot, pressure and tackle has a pitch location.

**Method in one line:** count actions per player → convert to per-90-minute rates → compare each player only with others in the same position group (percentiles) → combine related metrics into simple role scores → find similar players with cosine similarity.

> Work through every cell yourself. You need to be able to explain each choice in an interview, and change it if someone challenges it.

## 1. Get the data
Run once. Downloads about 155 MB into a `data` folder next to this notebook. Needs `requests` (`pip install requests pandas numpy matplotlib`).

In [ ]:
import json, math
from pathlib import Path
import requests

DATA = Path("data")
BASE = "https://raw.githubusercontent.com/statsbomb/open-data/master/data/"
COMP, SEASON = 55, 282   # UEFA Euro, 2024

def fetch(rel):
    path = DATA / rel
    if path.exists():
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    r = requests.get(BASE + rel, timeout=60); r.raise_for_status()
    path.write_bytes(r.content)

fetch(f"matches/{COMP}/{SEASON}.json")
match_ids = [m["match_id"] for m in json.load(open(DATA / f"matches/{COMP}/{SEASON}.json"))]
for i, mid in enumerate(match_ids, 1):
    fetch(f"events/{mid}.json"); fetch(f"lineups/{mid}.json")
    print(f"\r{i}/{len(match_ids)} matches", end="")

## 2. Load matches, events and line-ups

In [ ]:
import numpy as np, pandas as pd

matches = json.load(open(DATA / "matches/55/282.json"))
def mmss(s):
    m, sec = s.split(":"); return int(m) + int(sec) / 60

events, lineups = {}, {}
for m in matches:
    mid = m["match_id"]
    events[mid] = json.load(open(DATA / f"events/{mid}.json"))
    lineups[mid] = json.load(open(DATA / f"lineups/{mid}.json"))

## 3. Minutes played and main position
StatsBomb line-ups list each spell a player had in a position (`from` / `to` match clock). A blank `to` means he finished the match, so we use the last event time. A player's **main position** is where he played the most minutes. Goalkeepers are left out because outfield metrics don't describe them.

In [ ]:
rows = []
for mid, teams in lineups.items():
    end_min = max(e["minute"] + e["second"] / 60 for e in events[mid])
    for team in teams:
        for p in team["lineup"]:
            for pos in p["positions"]:
                start = mmss(pos["from"])
                stop = mmss(pos["to"]) if pos["to"] else end_min
                rows.append({"match_id": mid, "player_id": p["player_id"],
                             "player": p.get("player_nickname") or p["player_name"],
                             "team": team["team_name"], "position": pos["position"],
                             "minutes": max(stop - start, 0)})
stints = pd.DataFrame(rows)
mins = stints.groupby(["player_id", "player", "team"], as_index=False)["minutes"].sum()
main_pos = (stints.groupby(["player_id", "position"])["minutes"].sum()
            .reset_index().sort_values("minutes", ascending=False)
            .drop_duplicates("player_id")[["player_id", "position"]])
players = mins.merge(main_pos, on="player_id")

GROUPS = {
    "Centre-back": ["Center Back", "Left Center Back", "Right Center Back"],
    "Full-back": ["Left Back", "Right Back", "Left Wing Back", "Right Wing Back"],
    "Midfielder": ["Center Defensive Midfield", "Left Defensive Midfield", "Right Defensive Midfield",
                   "Center Midfield", "Left Center Midfield", "Right Center Midfield",
                   "Center Attacking Midfield", "Left Attacking Midfield", "Right Attacking Midfield"],
    "Winger": ["Left Wing", "Right Wing", "Left Midfield", "Right Midfield"],
    "Striker": ["Center Forward", "Left Center Forward", "Right Center Forward", "Secondary Striker"],
}
pos_to_group = {p: g for g, ps in GROUPS.items() for p in ps}
players["group"] = players["position"].map(pos_to_group)  # goalkeepers become NaN
players.head()

## 4. Count the actions
Definitions (write these in your own words, you will be asked):
- **Open-play passes only**: set pieces (throw-ins, corners, free kicks, goal kicks, kick-offs) are excluded.
- **Progressive pass / carry**: ends at least 25% closer to the centre of the opponent's goal. Carries must also gain 5+ units, so tiny touches don't count.
- **Passes into box**: complete passes ending in the penalty area that started outside it.
- **xA**: the xG of the shot a player's pass set up, credited to the passer.
- **npxG**: expected goals excluding penalties.
- **Tackles + interceptions**: successful ones only.

In [ ]:
GOAL = (120, 40)
def dist(loc): return math.hypot(GOAL[0] - loc[0], GOAL[1] - loc[1])
def progressive(start, end):
    """Moves the ball at least 25% closer to the centre of the opponent's goal."""
    return dist(end) <= 0.75 * dist(start)
def in_box(loc): return loc[0] >= 102 and 18 <= loc[1] <= 62

stats = {}
def add(pid, key, val=1):
    stats.setdefault(pid, {}); stats[pid][key] = stats[pid].get(key, 0) + val

for mid, evs in events.items():
    by_id = {e["id"]: e for e in evs}
    for e in evs:
        if "player" not in e: continue
        pid, t = e["player"]["id"], e["type"]["name"]
        loc = e.get("location")
        if t == "Pass":
            pa = e["pass"]
            if pa.get("type", {}).get("name") in ("Throw-in", "Corner", "Free Kick", "Kick Off", "Goal Kick"):
                continue  # open-play passes only
            add(pid, "passes_attempted")
            if "outcome" not in pa:  # no outcome = completed
                add(pid, "passes_completed")
                end = pa["end_location"]
                if progressive(loc, end): add(pid, "progressive_passes")
                if loc[0] < 80 <= end[0]: add(pid, "passes_into_final_third")
                if in_box(end) and not in_box(loc): add(pid, "passes_into_box")
            if pa.get("shot_assist") or pa.get("goal_assist"): add(pid, "key_passes")
        elif t == "Carry":
            end = e["carry"]["end_location"]
            if progressive(loc, end) and dist(loc) - dist(end) >= 5: add(pid, "progressive_carries")
        elif t == "Shot":
            sh = e["shot"]
            if sh["type"]["name"] == "Penalty": continue
            add(pid, "shots"); add(pid, "npxg", sh["statsbomb_xg"])
            if sh["outcome"]["name"] == "Goal": add(pid, "np_goals")
            kp = sh.get("key_pass_id")
            if kp and kp in by_id and "player" in by_id[kp]:
                add(by_id[kp]["player"]["id"], "xa", sh["statsbomb_xg"])
        elif t == "Dribble":
            if e["dribble"]["outcome"]["name"] == "Complete": add(pid, "dribbles_completed")
        elif t == "Pressure":
            add(pid, "pressures")
            if loc and loc[0] >= 80: add(pid, "pressures_final_third")
            if e.get("counterpress"): add(pid, "counterpresses")
        elif t == "Duel" and e["duel"]["type"]["name"] == "Tackle":
            if e["duel"].get("outcome", {}).get("name") in ("Won", "Success", "Success In Play", "Success Out"):
                add(pid, "tackles_won")
        elif t == "Interception":
            if e["interception"]["outcome"]["name"] in ("Won", "Success", "Success In Play", "Success Out"):
                add(pid, "interceptions")
        elif t == "Ball Recovery":
            if not e.get("ball_recovery", {}).get("recovery_failure"): add(pid, "ball_recoveries")
        elif t == "Clearance":
            add(pid, "clearances")

counts = pd.DataFrame.from_dict(stats, orient="index").fillna(0)
counts.index.name = "player_id"
df = players.merge(counts.reset_index(), on="player_id", how="left")
df[list(counts.columns)] = df[list(counts.columns)].fillna(0)

## 5. Per 90 and minimum minutes
Totals reward players who played more, so everything is per 90 minutes. Small samples give extreme rates (one great sub appearance), so we keep players with **270+ minutes**, roughly three full matches.

In [ ]:
MIN_MINUTES = 270  # three full matches
COUNT_COLS = [c for c in counts.columns if c != "passes_attempted"]
df = df[(df["minutes"] >= MIN_MINUTES) & df["group"].notna()].copy()
df["pass_completion"] = df["passes_completed"] / df["passes_attempted"].replace(0, np.nan)
for c in COUNT_COLS:
    df[c + "_p90"] = df[c] / df["minutes"] * 90
df["tackles_interceptions_p90"] = df["tackles_won_p90"] + df["interceptions_p90"]

METRICS = ["passes_completed_p90", "pass_completion", "progressive_passes_p90", "progressive_carries_p90",
           "passes_into_final_third_p90", "passes_into_box_p90", "key_passes_p90", "xa_p90",
           "shots_p90", "npxg_p90", "dribbles_completed_p90", "pressures_p90", "pressures_final_third_p90",
           "counterpresses_p90", "tackles_interceptions_p90", "ball_recoveries_p90", "clearances_p90"]
print(len(df), 'players'); df['group'].value_counts()

## 6. Percentiles within position group
A centre-back making 4 tackles a game and a winger making 4 tackles a game mean very different things. So each player is ranked only against his own group: 80th percentile = better than 80% of players in that group.

In [ ]:
for m in METRICS:
    df[m + "_pct"] = df.groupby("group")[m].rank(pct=True) * 100

## 7. Role scores
Each role score is the average of a few related percentiles. These groupings are **judgement calls**, not facts. A coach might define a "ball-playing centre-back" differently. That is a conversation worth having, not a weakness.

In [ ]:
ROLES = {
    "Centre-back": {"Ball-playing": ["progressive_passes_p90", "passes_completed_p90", "pass_completion"],
                    "Defending": ["tackles_interceptions_p90", "clearances_p90", "ball_recoveries_p90"]},
    "Full-back": {"Progressor": ["progressive_passes_p90", "progressive_carries_p90", "passes_into_final_third_p90"],
                  "Attacking": ["key_passes_p90", "xa_p90", "passes_into_box_p90"],
                  "Defending": ["tackles_interceptions_p90", "pressures_p90", "ball_recoveries_p90"]},
    "Midfielder": {"Deep playmaker": ["passes_completed_p90", "progressive_passes_p90", "pass_completion"],
                   "Ball winner": ["tackles_interceptions_p90", "pressures_p90", "ball_recoveries_p90"],
                   "Attacking": ["key_passes_p90", "xa_p90", "npxg_p90", "progressive_carries_p90"]},
    "Winger": {"Creator": ["key_passes_p90", "xa_p90", "passes_into_box_p90"],
               "Carrier": ["progressive_carries_p90", "dribbles_completed_p90"],
               "Goal threat": ["npxg_p90", "shots_p90"],
               "Presser": ["pressures_final_third_p90", "counterpresses_p90"]},
    "Striker": {"Goal threat": ["npxg_p90", "shots_p90"],
                "Link-up": ["key_passes_p90", "xa_p90", "passes_completed_p90"],
                "Presser": ["pressures_final_third_p90", "counterpresses_p90"]},
}
role_rows = []
for _, r in df.iterrows():
    for role, ms in ROLES[r["group"]].items():
        role_rows.append({"player": r["player"], "team": r["team"], "group": r["group"], "role": role,
                          "score": np.mean([r[m + "_pct"] for m in ms])})
roles = pd.DataFrame(role_rows)

def top(group, role, n=5):
    return roles[(roles.group == group) & (roles.role == role)].sort_values("score", ascending=False).head(n)

top("Midfielder", "Deep playmaker")

In [ ]:
top("Winger", "Creator")

## 8. Who plays like whom?
Each player becomes a vector of z-scores across all 17 metrics (how far above or below the group average he is). **Cosine similarity** compares the *shape* of two profiles: 1 = identical style, 0 = unrelated.

In [ ]:
def similar(player, n=5):
    row = df[df["player"] == player].iloc[0]
    g = df[df["group"] == row["group"]]
    z = (g[METRICS] - g[METRICS].mean()) / g[METRICS].std(ddof=0)
    z = z.fillna(0)
    v = z.loc[row.name].values
    sims = z.values @ v / (np.linalg.norm(z.values, axis=1) * np.linalg.norm(v) + 1e-9)
    out = g.assign(similarity=sims)[["player", "team", "position", "minutes", "similarity"]]
    return out[out["player"] != player].sort_values("similarity", ascending=False).head(n)

similar("Rodri")

## 9. Charts
A percentile profile for one player, and a scatter comparing two skills across a group.

In [ ]:
import matplotlib.pyplot as plt
MIN_MINUTES = 270
LABELS = {"passes_completed_p90": "Passes completed", "pass_completion": "Pass completion %",
  "progressive_passes_p90": "Progressive passes", "progressive_carries_p90": "Progressive carries",
  "passes_into_final_third_p90": "Passes into final third", "passes_into_box_p90": "Passes into box",
  "key_passes_p90": "Key passes", "xa_p90": "xA (expected assists)", "shots_p90": "Shots",
  "npxg_p90": "Non-penalty xG", "dribbles_completed_p90": "Dribbles completed",
  "pressures_p90": "Pressures", "pressures_final_third_p90": "Pressures in final third",
  "counterpresses_p90": "Counter-presses", "tackles_interceptions_p90": "Tackles + interceptions",
  "ball_recoveries_p90": "Ball recoveries", "clearances_p90": "Clearances"}
SECTIONS = [("In possession", ["passes_completed_p90", "pass_completion", "progressive_passes_p90",
             "progressive_carries_p90", "passes_into_final_third_p90"]),
            ("Chance creation & threat", ["passes_into_box_p90", "key_passes_p90", "xa_p90",
             "dribbles_completed_p90", "shots_p90", "npxg_p90"]),
            ("Out of possession", ["pressures_p90", "pressures_final_third_p90", "counterpresses_p90",
             "tackles_interceptions_p90", "ball_recoveries_p90"])]
COLS = ["#2B6CB0", "#C05621", "#2F855A"]

def profile(player, path):
    r = df[df.player == player].iloc[0]
    fig, ax = plt.subplots(figsize=(8, 7.5))
    y, ticks, labels = 0, [], []
    for (title, ms), col in zip(SECTIONS, COLS):
        ax.text(-2, y - 0.2, title.upper(), fontsize=9, fontweight="bold", color=col, ha="left")
        y -= 1
        for m in ms:
            v = r[m + "_pct"]
            ax.barh(y, v, color=col, alpha=0.85, height=0.7)
            ax.text(v + 1.5, y, f"{v:.0f}", va="center", fontsize=9)
            ticks.append(y); labels.append(LABELS[m]); y -= 1
        y -= 0.4
    ax.set_yticks(ticks, labels, fontsize=9)
    ax.set_xlim(0, 105); ax.set_xlabel(f"Percentile vs {r['group'].lower()}s at Euro 2024 (min {MIN_MINUTES} mins)")
    ax.axvline(50, color="#888", lw=0.8, ls="--")
    for s in ("top", "right"): ax.spines[s].set_visible(False)
    ax.set_title(f"{player} ({r['team']}), {r['position']}\n{r['minutes']:.0f} minutes, per-90 percentiles",
                 fontsize=12, loc="left")
    fig.text(0.01, 0.005, "Data: StatsBomb open data, UEFA Euro 2024", fontsize=7, color="#666")
    fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)

def scatter(group, x, y, path, label_top=8):
    g = df[df.group == group]
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(g[x], g[y], s=40, color="#2B6CB0", alpha=0.7)
    ax.axvline(g[x].median(), color="#999", lw=0.8, ls="--"); ax.axhline(g[y].median(), color="#999", lw=0.8, ls="--")
    lab = g.assign(s=g[x].rank() + g[y].rank()).sort_values("s", ascending=False).head(label_top)
    for _, r in lab.iterrows(): ax.annotate(r.player, (r[x], r[y]), fontsize=8, xytext=(4, 3), textcoords="offset points")
    ax.set_xlabel(LABELS[x] + " per 90"); ax.set_ylabel(LABELS[y] + " per 90")
    ax.set_title(f"{group}s at Euro 2024: {LABELS[x]} vs {LABELS[y]}", loc="left")
    for s in ("top", "right"): ax.spines[s].set_visible(False)
    fig.text(0.01, 0.005, "Data: StatsBomb open data. Dashed lines = group median.", fontsize=7, color="#666")
    fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)

In [ ]:
profile("Lamine Yamal", "profile.png")
scatter("Full-back", "progressive_passes_p90", "xa_p90", "fullbacks.png")

## 10. Export for Power BI
One wide table (one row per player) and one long table (one row per player per role), which suits slicers in Power BI.

In [ ]:
Path("outputs").mkdir(exist_ok=True)
df.round(3).to_csv("outputs/players_per90.csv", index=False)
roles.round(1).to_csv("outputs/role_scores.csv", index=False)

## 11. Your turn (do these before you put this on your CV)
1. **Pick a player you'd recommend** in one position. Write three sentences a coach could act on: what he's good at, what he isn't, and how sure you are (minutes played, opponents faced).
2. **Change a choice and see what moves.** Try 180 or 360 minimum minutes, or a different progressive-pass rule. Which rankings are stable and which flip?
3. **Challenge a role.** Rebuild one role score with different metrics. Defend your version in two lines.
4. **Find a weakness in the data.** Tournament football is 3–7 matches per team against very different opponents. How would you adjust for opponent strength or game state?
5. **Make it academy-shaped.** Man City's academy cares about development over time. Sketch how you'd track the same profile across a season for an U18 player, and what you'd show a coach each month.
6. **Build the Power BI page** from the two CSVs: a player selector, the percentile profile, and a similar-players table.

### Limitations to say out loud
- Small sample: one tournament, at most 7 matches per player.
- No opponent or game-state adjustment.
- Percentiles in small groups (16 strikers, 20 wingers) move in big steps.
- Event data misses off-ball work such as positioning and runs.